# CS604 Colab — Qian: last-block attention + cosine head @448

Full-data run: train only the last block’s QKV and output projection plus the prototype cosine classifier. Other backbone biases, LayerNorms and embeddings stay frozen. All 450,000 training images; fresh pretrained weights; no holdout or pilot checkpoint.

Attention LR 1e-4, head LR 3e-3, weight decay 0.05 (excluding 1D parameters), label smoothing 0.1, gradient clipping 1.0. Batch 64 × accumulation 4 = effective batch 256. BF16, seed 42, 144 training minutes, 5% time warmup + cosine. A100 40GB recommended. This is an unvalidated combined recipe, not an isolated ablation.

Reuse validated all-data prototypes_448.pt, or build them if absent. After training, set RUN_NAME to the printed run folder and predict last.pt with crop 0.75 + horizontal-flip TTA.


In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "qq_branch_v2"
METHOD = "lastblock_attention_cosine_448"
TEAM = "08"
PROTO_PT = f"{DRIVE_OUT}/prototypes_448.pt"
# Reuse only when provenance matches; otherwise explicitly rebuild.
FORCE_REBUILD_PROTOTYPES = False
RESOLVED_CONFIG = "/content/qq_attention_final_training_config.yaml"
EVAL_CROP_PCT = 0.75
TTA_FLIP = True


## 1. Clone or pull the repo

In [ ]:
import base64
import os
import subprocess
from getpass import getpass

# Pass credentials through a temporary child-process environment, not the URL.
token = getpass("GitHub PAT (hidden; repo scope). Leave empty for a public repo: ")
git_env = os.environ.copy()
if token:
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraHeader",
                   GIT_CONFIG_VALUE_0=f"Authorization: Basic {auth}")
    del auth
try:
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], env=git_env, check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], env=git_env, check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], env=git_env, check=True)
finally:
    del token, git_env
os.chdir(REPO_DIR)
actual = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
assert actual == BRANCH, f"Wrong branch: {actual}"
print("Branch:", actual)
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

## 4. Download data to `/content/data`

In [ ]:
# Stream subprocess output into the notebook, and stop on failure.
def run_live(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)
    except BaseException:
        process.terminate()
        process.wait()
        raise

run_live(["python", "-u", "scripts/download_data.py", "--dest", DATA_DIR])


## 4b. Build all-data prototypes at 448

Use all 45 images per class. Reuse only a file whose metadata matches the backbone, 448px resolution, full training manifest and file hash. Prototype extraction is outside the 144-minute training timer.


In [ ]:
import hashlib
import json
from collections import Counter
from pathlib import Path

import torch
from src.utils import load_config, apply_overrides, save_config

assert torch.cuda.is_available(), "Select a GPU runtime before prototype extraction."

def file_hash(path):
    checksum = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            checksum.update(chunk)
    return checksum.hexdigest()

def resolve_training_config():
    cfg = load_config(f"configs/{METHOD}.yaml")
    apply_overrides(cfg, data_root=DATA_DIR, output_dir=DRIVE_OUT)
    if cfg.get("head_init") == "prototype":
        cfg.prototype_file = PROTO_PT
    save_config(cfg, RESOLVED_CONFIG)
    return cfg

cfg = resolve_training_config()
manifest = Path(DATA_DIR) / "train.txt"
rows = [line.split() for line in manifest.read_text().splitlines() if line.strip()]
assert all(len(row) == 2 for row in rows), "Training manifest must contain path and label."
counts = Counter(int(row[1]) for row in rows)
assert len({row[0] for row in rows}) == len(rows), "Duplicate training image paths."
assert set(counts) == set(range(int(cfg.num_classes))), "Training manifest is missing classes."
assert all(count == 45 for count in counts.values()), "Expected all 45 images per class."
assert len(rows) == 450000, "Expected the complete 450,000-image training set."
print("Training set:", len(rows), "images;", len(counts), "classes; 45 images/class")

prototype_path = Path(PROTO_PT)
metadata_path = prototype_path.with_suffix(".metadata.json")
expected = {"model_name": cfg.model_name, "img_size": int(cfg.img_size),
            "num_classes": int(cfg.num_classes), "training_images": len(rows),
            "train_manifest_sha256": file_hash(manifest)}

def validate_prototypes(config):
    blob = torch.load(prototype_path, map_location="cpu", weights_only=True)
    assert blob.get("model_name") == config.model_name, "Prototype backbone does not match."
    assert tuple(blob["prototypes"].shape) == (int(config.num_classes), 768)
    assert torch.isfinite(blob["prototypes"]).all(), "Nonfinite prototype features."
    assert torch.allclose(blob["prototypes"].float().norm(dim=1),
                          torch.ones(int(config.num_classes)), atol=1e-4), "Prototypes must be normalized."
    assert tuple(blob["samples_per_class"].shape) == (int(config.num_classes),)
    assert torch.all(blob["samples_per_class"] == 45), "Prototypes must use all 45 images/class."
    actual_hash = file_hash(prototype_path)
    metadata = json.loads(metadata_path.read_text())
    required = {**expected, "model_name": config.model_name, "img_size": int(config.img_size),
                "num_classes": int(config.num_classes),
                "train_manifest_sha256": file_hash(Path(config.data_root) / "train.txt")}
    assert all(metadata.get(key) == value for key, value in required.items()), \
        "Prototype backbone, resolution or full training manifest does not match."
    assert metadata.get("prototype_sha256") == actual_hash, "Prototype file was changed."
    print("Validated prototype contents:", prototype_path)

if cfg.get("head_init") == "prototype":
    if FORCE_REBUILD_PROTOTYPES or not prototype_path.exists():
        # Explicitly disable any per-class sampling inherited from another recipe.
        proto_cfg = load_config(RESOLVED_CONFIG)
        proto_cfg.prototype_per_class = None
        proto_config_path = "/content/qq_all_data_prototypes_config.yaml"
        save_config(proto_cfg, proto_config_path)
        run_live(["python", "-u", "scripts/build_prototypes.py", "--config", proto_config_path,
                  "--data_root", DATA_DIR, "--output", PROTO_PT, "--batch_size", str(cfg.batch_size)])
        metadata = {**expected, "prototype_sha256": file_hash(prototype_path)}
        metadata_path.write_text(json.dumps(metadata, indent=2))
    elif not metadata_path.exists():
        raise ValueError("Prototype provenance is missing. Set FORCE_REBUILD_PROTOTYPES=True and rerun 4b.")
    validate_prototypes(cfg)
else:
    print("This configuration does not use prototype initialization.")


## Review settings and trainable parameters

Check that only final-block QKV/output projection and the classifier train. Backbone adaptation must be 2,362,368 parameters (about 2.74% at 448), strictly below 5% with the classifier excluded.


In [ ]:
import pandas as pd
from src.model import build_model

methods = ["lastblock_attention_cosine_448", "lastblock_attention", "full_ft", "linear_probe", "lora_r8"]
merged = {name: load_config(f"configs/{name}.yaml").to_dict() for name in methods}
display(pd.DataFrame(merged).fillna(""))
chosen = resolve_training_config()
display(pd.DataFrame(list(chosen.to_dict().items()), columns=["parameter", "value"]))

# Count the architecture without downloading pretrained weights or allocating GPU memory.
count_cfg = load_config(RESOLVED_CONFIG)
count_cfg.pretrained = False
count_model = build_model(count_cfg)
head_ids = {id(p) for p in count_model.get_classifier().parameters()}
backbone_parameters = sum(p.numel() for name, p in count_model.named_parameters()
                          if id(p) not in head_ids and not name.endswith(("prefix_key", "prefix_value")))
trainable_backbone = sum(p.numel() for p in count_model.parameters()
                        if p.requires_grad and id(p) not in head_ids)
PARAMETER_STATS = {"original_backbone_parameters": backbone_parameters,
                   "trainable_backbone_parameters": trainable_backbone,
                   "trainable_backbone_pct": 100 * trainable_backbone / backbone_parameters,
                   "trainable_classifier_parameters": sum(p.numel() for p in count_model.get_classifier().parameters()
                                                          if p.requires_grad)}
# Verify exact scope: only final-block QKV/proj and the classifier train.
expected_ids = {id(p) for module in (count_model.blocks[-1].attn.qkv,
                                   count_model.blocks[-1].attn.proj)
                for p in module.parameters()}
assert all(p.requires_grad == (id(p) in expected_ids)
           for _, p in count_model.named_parameters() if id(p) not in head_ids)
assert trainable_backbone == 2362368
assert PARAMETER_STATS["trainable_backbone_pct"] < 5
print("Parameter counts (classifier excluded from cap):", PARAMETER_STATS)
del count_model


## 5. Train on all data for 144 minutes

The budget counts training time; setup, prototype extraction and saving are outside it. No epoch ceiling and no labeled holdout. Save the printed run folder for RUN_NAME below. Checkpoints contain trainable parameters and require the original pretrained backbone when loading.


In [ ]:
assert torch.cuda.is_available(), "Select an A100 GPU runtime."
check = resolve_training_config()
print("GPU:", torch.cuda.get_device_name(0))
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("Resolved training config:", check.to_dict())

assert check.method == "last_blocks" and check.pretrained
assert check.last_n_blocks == 1 and check.train_scope == "attention"
assert check.img_size == 448 and check.num_classes == 10000
assert check.cosine_scale == 20.0
assert check.batch_size == 64 and check.grad_accum_steps == 4
assert check.lr == 1e-4 and check.head_lr == 3e-3
assert check.weight_decay == 0.05 and check.label_smoothing == 0.1
assert check.clip_grad_norm == 1.0 and check.seed == 42 and check.bf16
assert check.epochs is None and check.max_train_minutes == 144
assert check.time_warmup_fraction == 0.05
assert check.head_type == "cosine" and check.head_init == "prototype"
assert not check.get("ablation") and not check.get("unfreeze_extra")
assert file_hash(Path(DATA_DIR) / "train.txt") == expected["train_manifest_sha256"]
validate_prototypes(check)

run_live(["python", "-u", "-m", "src.train", "--config", RESOLVED_CONFIG])


## 6. Predict with crop 0.75 + horizontal-flip TTA

Set RUN_NAME to the final training folder. Use its saved config and last.pt. Average probabilities from original and mirrored images, then write the submission TXT/ZIP into the run folder. Test prediction runs only if test data is available.


In [ ]:
from pathlib import Path
import subprocess

RUN_NAME = "REPLACE_WITH_RUN_DIR"
RUN_DIR = Path(DRIVE_OUT) / RUN_NAME
CKPT = str(RUN_DIR / "last.pt")
assert Path(CKPT).is_file(), f"Checkpoint missing: {CKPT}"

# Use the saved settings from this exact run.
saved_cfg = load_config(RUN_DIR / "config.yaml")
if METHOD == "lastblock_attention_cosine_448":
    assert saved_cfg.method == "last_blocks" and saved_cfg.img_size == 448
    assert saved_cfg.last_n_blocks == 1 and saved_cfg.train_scope == "attention"
    assert saved_cfg.grad_accum_steps == 4
    assert saved_cfg.lr == 1e-4 and saved_cfg.head_lr == 3e-3
    assert saved_cfg.head_type == "cosine"
    assert saved_cfg.epochs is None and saved_cfg.max_train_minutes == 144
    assert saved_cfg.data_root == DATA_DIR, "Select the all-data final run, not a split experiment."
if "PARAMETER_STATS" in globals():
    (RUN_DIR / "parameter_stats.json").write_text(json.dumps(PARAMETER_STATS, indent=2))
def predict(split):
    command = [
        "python", "-m", "src.predict",
        "--config", str(RUN_DIR / "config.yaml"),
        "--ckpt", CKPT, "--split", split, "--team", TEAM,
        "--data_root", DATA_DIR, "--out_dir", str(RUN_DIR),
        "--crop_pct", str(EVAL_CROP_PCT),
    ]
    if TTA_FLIP:
        command.append("--tta_flip")
    subprocess.run(command, check=True)

(RUN_DIR / "prediction_settings.json").write_text(json.dumps(
    {"checkpoint": CKPT, "crop_pct": EVAL_CROP_PCT, "tta_flip": TTA_FLIP}, indent=2))
predict("val")
__import__("shutil").make_archive(str(RUN_DIR / f"val_team_{TEAM}"), "zip", root_dir=str(RUN_DIR), base_dir=f"val_team_{TEAM}.txt")
if (Path(DATA_DIR) / "test.txt").is_file() and (Path(DATA_DIR) / "test").is_dir():
    predict("test")
else:
    print("Test data not available; skipping test prediction.")


## 7. Loss curves

Plot the losses recorded during training, after training finishes. Save and display the plots in this experiment’s Drive folder. Validation curves require labeled validation.


In [ ]:
from IPython.display import display, Image

FIGURE_DIR = RUN_DIR / "figures"
subprocess.run([
    "python", "scripts/plot_curves.py",
    "--run_dir", str(RUN_DIR), "--out_dir", str(FIGURE_DIR),
], check=True)

for plot in sorted(FIGURE_DIR.glob("*.png")):
    print(plot.name)
    display(Image(filename=str(plot)))


## 8. Inference latency

Load the trained checkpoint selected in section 6. Measure forward-pass latency on synthetic images at batch sizes 1 and 64, excluding preprocessing and transfers. Use 30 warm-ups and 200 timed iterations. Save results to `inference_latency.json`. This runs outside the training timer; keep GPU and measurement settings consistent between experiments.


In [ ]:
import json
import torch

from src.utils import load_config, apply_overrides, set_seed
from src.model import build_model
from src.predict import load_checkpoint
from scripts.benchmark_latency import measure

assert torch.cuda.is_available(), "A GPU runtime is required."
cfg = load_config(RUN_DIR / "config.yaml")
# Partial PEFT checkpoints require the original pretrained backbone.
checkpoint_info = torch.load(CKPT, map_location="cpu", weights_only=False)
partial = bool(checkpoint_info.get("trainable_only", False))
del checkpoint_info
apply_overrides(cfg, pretrained=partial)
set_seed(int(cfg.seed))

device = torch.device("cuda")
model = build_model(cfg).to(device)
load_checkpoint(model, CKPT, device)
model.eval()

warmup = 30
iterations = 200
use_bf16 = bool(cfg.bf16)
results = {
    "checkpoint": CKPT,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_capacity_GiB": torch.cuda.get_device_properties(0).total_memory / 1024**3,
    "torch_version": str(torch.__version__),
    "precision": "bf16" if use_bf16 else "fp32",
    "img_size": int(cfg.img_size),
    "warmup_iterations": warmup,
    "timed_iterations": iterations,
    "scope": "Single-view forward pass on synthetic inputs; excludes flip TTA, loading, preprocessing and transfers",
    "measurements": [],
}

try:
    for batch_size in (1, 64):
        batch = torch.randn(
            batch_size, 3, int(cfg.img_size), int(cfg.img_size), device=device,
        )
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_bf16):
            ms_per_image, _ = measure(model, batch, device, warmup, iterations)
        results["measurements"].append({
            "batch_size": batch_size,
            "ms_per_image": ms_per_image,
            "ms_per_batch": ms_per_image * batch_size,
        })
        del batch
finally:
    del model
    torch.cuda.empty_cache()

output = RUN_DIR / "inference_latency.json"
output.write_text(json.dumps(results, indent=2), encoding="utf-8")
print(json.dumps(results, indent=2))
print("Saved:", output)
